# 第 24 天：Barra基础

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：Barra基础
> 必做：风格因子
> 选做：行业因子
> 目标产出：Barra笔记

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 理解风格因子、行业因子、特质收益的分解方式。
2. 构建一个简化版风险暴露矩阵。
3. 用横截面回归估计风格收益和行业收益。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

Barra 模型像一张股票的体检表：它不只问这只股票涨不涨，还问它暴露在哪些风格、属于哪个行业、承担了哪些共同风险。

## 5. 今日核心实验


### 实验 1：构建 Barra 风格暴露矩阵

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
style_names = ["size", "value", "momentum_20", "low_vol", "liquidity"]
exposure_date = dates[-80]

style_exposure = pd.DataFrame({
    name: factor_library[name].loc[exposure_date]
    for name in style_names
})

industry_exposure = pd.get_dummies(industries)
industry_exposure = industry_exposure.drop(columns=industry_exposure.columns[-1])

X = pd.concat([style_exposure, industry_exposure], axis=1).astype(float)
y = returns.shift(-1).loc[exposure_date].reindex(X.index)
valid = y.notna() & X.notna().all(axis=1)

print("暴露矩阵形状：", X.loc[valid].shape)
print(X.loc[valid].head().round(3))


### 实验 2：横截面回归：估计当天风格和行业收益

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
X_valid = X.loc[valid]
y_valid = y.loc[valid]
X_design = np.column_stack([np.ones(len(X_valid)), X_valid.to_numpy()])
coef = np.linalg.lstsq(X_design, y_valid.to_numpy(), rcond=None)[0]

factor_return = pd.Series(coef[1:], index=X_valid.columns, name=exposure_date)
residual = y_valid - X_design @ coef
r2 = 1 - residual.var() / y_valid.var()

print("风格/行业收益估计：")
print(factor_return.round(5))
print("\n横截面 R2：", round(float(r2), 4))


### 实验 3：暴露体检：标准化、共线性和解释性

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
exposure_health = pd.DataFrame({
    "mean": X_valid.mean(),
    "std": X_valid.std(),
    "missing": X_valid.isna().mean(),
})

condition_number = np.linalg.cond(X_design)
print(exposure_health.round(4))
print("\n设计矩阵条件数：", round(float(condition_number), 2))


### 实验 4：风格收益时间序列：不是只看一天

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
factor_return_rows = []
for dt in dates[80:-2]:
    style_exp = pd.DataFrame({name: factor_library[name].loc[dt] for name in style_names})
    ind_exp = industry_exposure.copy()
    X_dt = pd.concat([style_exp, ind_exp], axis=1).astype(float)
    y_dt = returns.shift(-1).loc[dt].reindex(X_dt.index)
    valid_dt = y_dt.notna() & X_dt.notna().all(axis=1)
    if valid_dt.sum() < X_dt.shape[1] + 5:
        continue
    design = np.column_stack([np.ones(valid_dt.sum()), X_dt.loc[valid_dt].to_numpy()])
    coef_dt = np.linalg.lstsq(design, y_dt.loc[valid_dt].to_numpy(), rcond=None)[0]
    factor_return_rows.append(pd.Series(coef_dt[1:], index=X_dt.columns, name=dt))

factor_returns_barra = pd.DataFrame(factor_return_rows)
print(factor_returns_barra[style_names].describe().round(5))


### 实验 5：Barra 笔记：风格因子的风险语言

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
style_cumulative = factor_returns_barra[style_names].cumsum()
style_cumulative.plot(figsize=(10, 4), title="简化 Barra 风格收益累计")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：Barra基础
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：把 Barra 当成选股 Alpha，而不是风险和归因框架。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：行业哑变量和截距同时使用时不处理共线性。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：暴露未标准化，导致回归系数难以解释。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：只看风格收益，不看残差和解释度。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 25 天会用 Barra 做收益归因和风险归因。

## 13. 一句话收尾

Barra基础 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
